In [1]:
import pandas as pd, glob, os

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
ROOT = "/content/drive/My Drive/Dugwell/data/yearly/timeseries"
files = glob.glob(os.path.join(ROOT, "*", "*", "*.csv"))
print(len(files), "files found")

frames, bad = [], []
for f in files:
    try:
        df = pd.read_csv(f, dtype={"station_id": str})
        df["state"]    = f.split(os.sep)[-3]
        df["district"] = f.split(os.sep)[-2]
        frames.append(df)
    except Exception as e:
        bad.append((f, str(e)))

merged = pd.concat(frames, ignore_index=True)
merged["station_id"] = merged["station_id"].str.strip()

season_cols = [f"{y}_season{i}" for y in (2024, 2025) for i in range(1, 5)]
for c in season_cols:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")   # keeps them Float in GEE

print("rows before dedupe:", len(merged))
merged = merged.drop_duplicates("station_id")
print("unique stations:", len(merged), "| unreadable files:", len(bad))
print(merged[["latitude","longitude"] + season_cols].isna().sum())

merged.to_csv("gw_2024_2025.csv", index=False)

500 files found


/tmp/ipykernel_7116/72968652.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(frames, ignore_index=True)


rows before dedupe: 8324
unique stations: 8324 | unreadable files: 0
latitude           0
longitude          0
2024_season1    3857
2024_season2    1950
2024_season3    1159
2024_season4    2201
2025_season1    2201
2025_season2    2566
2025_season3    3482
2025_season4    8324
dtype: int64


In [5]:
import pandas as pd
m = pd.read_csv("/content/gw_2024_2025.csv", dtype={"station_id": str})
s = m["station_id"]

print("numeric-only:", s.str.fullmatch(r"\d+").sum())
print("alphanumeric:", (~s.str.fullmatch(r"\d+")).sum())
print("lengths of numeric ids:\n", s[s.str.fullmatch(r"\d+")].str.len().value_counts())
print("ids ending in '.0' or containing 'e':", s.str.contains(r"\.0$|[eE]\+").sum())
print(s[s.str.fullmatch(r"\d+")].head(10).tolist())

numeric-only: 3216
alphanumeric: 5108
lengths of numeric ids:
 station_id
15    3216
Name: count, dtype: int64
ids ending in '.0' or containing 'e': 0
['222816083443901', '222757083571801', '223240083442002', '225835083391501', '225235083391501', '225058084085201', '224110083513001', '224110083513002', '223217083510202', '223600083404501']
